# LangChain Agent without FastAPI - Core Concepts
This notebook strips away the FastAPI framework to focus purely on the core components of the LangChain agent. This helps illustrate how the internal logic works independently of a web API framework.

In [ ]:
# Run this cell to install the required dependencies
!pip install langchain-openai pydantic-settings

In [ ]:
import os
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    app_name: str = "Core LangChain Agent"
    
    # Please replace this with your actual OpenAI API key, or use a .env file
    openai_api_key: str = "sk-your-api-key-here" 
    
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8")

# Instantiate settings to be used across the app
settings = Settings()

In [ ]:
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    """What is sent to the agent."""
    query: str = Field(..., description="The question or prompt for the agent.")

class ChatResponse(BaseModel):
    """What the agent returns."""
    answer: str = Field(..., description="The response generated by the LangChain agent.")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

class SimpleAgentGenerator:
    def __init__(self, api_key: str):
        # Initialize the LLM
        self.llm = ChatOpenAI(
            temperature=0.7, 
            openai_api_key=api_key,
            model="gpt-3.5-turbo"
        )
        
        # Define the system's persona and prompt
        self.prompt = PromptTemplate(
            input_variables=["query"],
            template="You are a highly intelligent and helpful tutor. Answer the student's query clearly and concisely.\n\nQuery: {query}"
        )
        
        # Create a simple chain
        self.chain = self.prompt | self.llm | StrOutputParser()

    def generate_response(self, query: str) -> str:
        try:
            return self.chain.invoke({"query": query})
        except Exception as e:
            return f"Error generating response: {str(e)}"

In [ ]:
class AgentService:
    def __init__(self):
        # Instantiate the generator using system configurations
        self.generator = SimpleAgentGenerator(api_key=settings.openai_api_key)

    def process_chat(self, request: ChatRequest) -> ChatResponse:
        """Processes the request and coordinates with the generator."""
        user_query = request.query
        llm_answer = self.generator.generate_response(user_query)
        return ChatResponse(answer=llm_answer)

In [ ]:
def main():
    print("--- Initializing Agent System ---")
    # 1. Initialize the service
    service = AgentService()
    print("System ready.\n")
    
    # 2. Simulate a user input
    user_input = "Explain the difference between synchronous and asynchronous execution."
    print(f"User Query: {user_input}\n")
    
    # 3. Create the Domain request object (Simulating what FastAPI does automatically)
    request = ChatRequest(query=user_input)
    
    # 4. Process the request through the service
    print("Agent thinking...\n")
    response = service.process_chat(request)
    
    # 5. Output the result
    print("--- Agent Response ---")
    print(response.answer)

if __name__ == "__main__":
    # Only run this if the API key is set, otherwise it will throw an error
    if settings.openai_api_key != "sk-your-api-key-here":
        main()
    else:
        print("Please set your OPENAI_API_KEY in the config section above to run this example.")